## Random Forest - Multi-Region Built-Up Classification

Goal: train a Random Forest classifier that generalizes across different
settlement types, not just Kasarani. Training on one neighborhood risks
biasing the model toward that area's specific spectral signature (e.g.
informal-settlement roofing) and misclassifying areas that look different
(e.g. formal, lower-density suburbs).

This notebook trains the model once, on a deliberately diverse sample of
Nairobi regions, then exports it as a reusable asset - it is not retrained
per query. Any future user-specified region is classified using this same
frozen model.

Reusable logic lives in `src/classification.py`, following the same
notebook-to-module pattern used elsewhere in this project.

In [1]:
import sys
sys.path.insert(0, '../src')

import importlib
import classification
importlib.reload(classification)

import ee
from classification import (
    get_region_boundary, get_sentinel2_composite, build_feature_image,
    get_worldcover_builtup, sample_region_points, train_random_forest,
    classify_builtup, FEATURE_NAMES,
)

ee.Initialize(project="riparian-encroachment")

### Step 1: Define diverse training regions

Six regions, chosen to deliberately span different settlement types:
informal/mixed (Kasarani), dense informal (Kibera), formal low-density
(Karen), formal medium-density (Kileleshwa), dense commercial (CBD), and a
non-built control (Nairobi National Park).

Some of these - Kibera and CBD, confirmed below - have no official
administrative boundary in OpenStreetMap (common for informal settlements
and less formally bounded areas). `get_region_boundary()` falls back to a
point + radius buffer in that case.

In [2]:
# Deliberately diverse — informal, formal, dense commercial, non-built control.
# Training only on Kasarani would bias the model toward informal-settlement
# spectral signatures and likely misclassify formal low-density suburbs.
TRAINING_REGIONS = {
    'kasarani':   ("Kasarani, Nairobi, Kenya", 3000),
    'kibera':     ("Kibera, Nairobi, Kenya", 1500),      # smaller — Kibera itself is compact
    'karen':      ("Karen, Nairobi, Kenya", None),        # has a real boundary, no fallback needed
    'kileleshwa': ("Kileleshwa, Nairobi, Kenya", None),
    'cbd':        ("Nairobi Central Business District, Kenya", 1500),
    'nnp':        ("Nairobi National Park, Kenya", None),
}

START_DATE, END_DATE = '2024-06-01', '2024-09-30'

### Step 2: Stratified sample from each region, then merge

For each region: build the Sentinel-2 composite, compute spectral features
(6 bands + NDVI + NDBI), pull WorldCover labels, and draw a stratified
sample (balanced built-up / non-built-up where possible). Samples from all
six regions are then merged into one training pool - this merge is what
actually produces the diverse training set, not anything in the training
step itself.

In [3]:
all_samples = []

for name, (place, radius) in TRAINING_REGIONS.items():
    if radius:
        region = get_region_boundary(place, fallback_radius_m=radius)
    else:
        region = get_region_boundary(place)
    composite, n_scenes = get_sentinel2_composite(region, START_DATE, END_DATE)
    features = build_feature_image(composite)
    worldcover_builtup = get_worldcover_builtup(region)

    samples = sample_region_points(features, worldcover_builtup, region, num_points=500)
    all_samples.append(samples)
    print(f"{name}: {n_scenes} scenes, {samples.size().getInfo()} sample points")

merged_samples = ee.FeatureCollection(all_samples).flatten()
print("Total training points across all regions:", merged_samples.size().getInfo())

kasarani: 6 scenes, 1000 sample points
  (no boundary polygon for 'Kibera, Nairobi, Kenya' — using 1500m point buffer instead)
kibera: 6 scenes, 1000 sample points
karen: 11 scenes, 1000 sample points
kileleshwa: 6 scenes, 1000 sample points
cbd: 6 scenes, 63 sample points
nnp: 6 scenes, 1000 sample points
Total training points across all regions: 5063


### Diagnostic - why CBD returned far fewer points than the other regions

Every other region returned ~1000 points; CBD returned only 63. Checked
directly: CBD is **100% built-up** per WorldCover, so `stratifiedSample`
had no non-built-up pixels to draw from - this is a real geographic fact,
not a bug. CBD's training/test contribution is built-up-only as a result.

In [4]:
cbd_region = get_region_boundary("Nairobi Central Business District, Kenya", fallback_radius_m=1500)
worldcover_builtup = get_worldcover_builtup(cbd_region)
frac = worldcover_builtup.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=cbd_region, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100
print(f"CBD built-up fraction (WorldCover): {frac:.1f}%")

CBD built-up fraction (WorldCover): 100.0%


### Step 3: 70/30 split, train, evaluate on held-out points

Same discipline as the earlier single-region model: the classifier is only
ever scored on the 30% of points it never trained on, so this number is a
fair, honest comparison - not inflated by testing on its own training data.

In [5]:
merged_samples = merged_samples.randomColumn('split', seed=42)
train_samples = merged_samples.filter(ee.Filter.lt('split', 0.7))
test_samples = merged_samples.filter(ee.Filter.gte('split', 0.7))

print("Train points:", train_samples.size().getInfo())
print("Test points:", test_samples.size().getInfo())

classifier = train_random_forest(train_samples)

test_accuracy = test_samples.classify(classifier).errorMatrix(
    'builtup', 'classification'
).accuracy().getInfo()
print(f"Held-out accuracy across diverse regions: {test_accuracy * 100:.1f}%")

Train points: 3577
Test points: 1486
Held-out accuracy across diverse regions: 85.9%


### Diagnostic - accuracy broken down by region

The overall held-out accuracy is a useful headline number, but it can hide
whether the model performs unevenly across settlement types - exactly the
bias risk this notebook exists to check. Re-running the held-out accuracy
separately per region surfaces that.

Note: CBD's 100% here is not a meaningful signal - with no non-built-up
points in its test set, there was nothing hard to get right. Treat it as
uncomparable to the other five regions' scores.

In [6]:
for name, (place, radius) in TRAINING_REGIONS.items():
    if radius:
        region = get_region_boundary(place, fallback_radius_m=radius)
    else:
        region = get_region_boundary(place)

    region_test = test_samples.filterBounds(region)
    n = region_test.size().getInfo()
    if n == 0:
        print(f"{name:12s}  n=   0  (no test points fell in this region)")
        continue
    acc = region_test.classify(classifier).errorMatrix(
        'builtup', 'classification'
    ).accuracy().getInfo()
    print(f"{name:12s}  n={n:4d}  accuracy={acc*100:.1f}%")

kasarani      n= 275  accuracy=85.8%
  (no boundary polygon for 'Kibera, Nairobi, Kenya' — using 1500m point buffer instead)
kibera        n= 308  accuracy=85.1%
karen         n= 323  accuracy=83.0%
kileleshwa    n= 275  accuracy=82.5%
cbd           n=  23  accuracy=100.0%
nnp           n= 318  accuracy=92.8%


### Step 4: Persist the trained classifier

Earth Engine classifiers live server-side and can't be pickled locally, so
they're persisted via `Export.classifier.toAsset()` instead. This is what
makes "train once, apply to any future region" possible - later sessions
load this saved asset directly (`ee.Classifier.load(asset_id)`) rather than
re-running the training pipeline above.

Requires the destination asset folder to already exist
(`projects/riparian-encroachment/assets`) - created once, out of band, via
`ee.data.createAsset({'type': 'FOLDER'}, ...)`.

> ee.data.createAsset({'type': 'FOLDER'}, 'projects/riparian-encroachment/assets')

But depends on your project name on EarthEngine

In [12]:
export_task = ee.batch.Export.classifier.toAsset(
    classifier=classifier,
    description='nairobi_builtup_rf_v1',
    assetId='projects/riparian-encroachment/assets/nairobi_builtup_rf_v1',
)
export_task.start()

import time
while export_task.active():
    print("Exporting classifier...", export_task.status()['state'])
    time.sleep(15)
print("Done:", export_task.status()['state'])

if export_task.status()['state'] == 'FAILED':
    print("Error:", export_task.status().get('error_message'))

Exporting classifier... READY
Done: COMPLETED
